# DisEnergy — Google Colab GPU Execution Suite (T4 / A100)
Runs the GPU-intensive components of the DisEnergy research plan using the exact frozen `data/splits/*.json` manifests from GitHub:
1. **GPU Compute Profiling & Bures-W2 ($w_s=2$) Ablation** (`profile_compute.py`)
2. **Regime 2 Label Scarcity (1%, 5%, 10%, 25%, 100%) & DisEnergy-Net Synergy** (`UNet`, `AttentionUNet`, `UNet++`, `SegFormer_B0`, `DisEnergyNet`)
3. **Zero-Shot Foundation Model Baselines** (`SAM ViT-B` / `MedSAM`, `DINOv2`) under clean & 10-15 dB noise stress
4. **ISIC 2018 -> PH2 OOD & MVTec AD Benchmarks** (+ optional BraTS 2D/3D MRI via Kaggle API)
5. **Unified Export** (`colab_outputs.zip`) ready to merge into `outputs/` and recompile `paper/main.pdf`.

In [ ]:
# 1. Clone repo & verify CUDA GPU
!git clone https://github.com/Vishal442005/DisEnergy.git
%cd DisEnergy
%env PYTHONPATH=/content/DisEnergy
!nvidia-smi
!pip install -q --no-deps -e .
!pip install -q gudhi POT scikit-posthocs nibabel SimpleITK pyyaml tqdm segment-anything transformers timm

In [ ]:
# 2. Verify all theorems & leak-free splits on GPU environment
!pytest -v
!python experiments/profile_compute.py --image-size 256 --device cuda --out-dir outputs/profiling_gpu

In [ ]:
# 3. Download public datasets on Colab GPU (ECSSD, BUSI-WHU-Seg, DRIVE, EuroSAT, PH2, MVTec)
!mkdir -p data/raw/ecssd data/raw/eurosat
!curl -sSL "http://www.cse.cuhk.edu.hk/leojia/projects/hsaliency/data/ECSSD/images.zip" -o data/raw/ecssd/images.zip
!curl -sSL "http://www.cse.cuhk.edu.hk/leojia/projects/hsaliency/data/ECSSD/ground_truth_mask.zip" -o data/raw/ecssd/ground_truth_mask.zip
!python data/prepare_datasets.py --dataset ecssd --image-size 256
!git clone --depth 1 "https://github.com/rohit9934/DRIVE-Digital-Retinal-Images-for-Vessel-Extraction.git" data/raw/drive
!python data/prepare_datasets.py --dataset drive --image-size 256
!git clone --depth 1 "https://github.com/vikaschouhan/PH2-dataset.git" data/raw/ph2_ood

In [ ]:
# 4. Optional: Download ISIC 2018 (~11.2 GB) & MVTec AD (~5.3 GB) on Colab SSD
# !mkdir -p data/raw/isic2018 data/raw/mvtec
# !curl -sSL "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task1-2_Training_Input.zip" -o data/raw/isic2018/input.zip
# !curl -sSL "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task1_Training_GroundTruth.zip" -o data/raw/isic2018/gt.zip
# !unzip -q data/raw/isic2018/input.zip -d data/raw/isic2018 && unzip -q data/raw/isic2018/gt.zip -d data/raw/isic2018
# !python data/prepare_datasets.py --dataset isic2018 --image-size 256
# !wget -q "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f282/download/420938113-1629952094/mvtec_anomaly_detection.tar.xz"
# !tar -xJf mvtec_anomaly_detection.tar.xz -C data/raw/mvtec
# !python data/prepare_datasets.py --dataset mvtec --image-size 256

In [ ]:
# 5. Run full-epoch Regime 2 Scarcity & DisEnergyNet benchmarks on GPU (FP16 AMP)
!python experiments/run_scarcity_and_hybrid.py --split-json data/splits/ecssd.json --out-dir outputs/ecssd_scarcity_gpu --epochs 25 --device cuda
!python experiments/run_scarcity_and_hybrid.py --split-json data/splits/drive.json --out-dir outputs/drive_scarcity_gpu --epochs 35 --device cuda

In [ ]:
# 6. Zero-shot SAM (ViT-B) Comparison under Clean & 10-15 dB Noise Stress
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
import json, numpy as np
from baselines.foundation import SAMZeroShotBaseline
from analysis.metrics import compute_segmentation_metrics
from disenergy.utils import inject_noise
sam = SAMZeroShotBaseline('sam_vit_b_01ec64.pth', model_type='vit_b', device='cuda')
m = json.loads(open('data/splits/ecssd.json').read())
id2r = {r['sample_id']: r for r in m['records']}
for snr in (None, 15.0, 10.0):
    dscs = []
    for sid in m['folds'][0]['test_ids'][:20]:
        img = np.load(id2r[sid]['image_path'])
        if snr is not None: img = inject_noise(img, 'speckle', snr_db=snr)
        gt = np.load(id2r[sid]['mask_path']).astype(bool)
        pred = sam.predict_mask((img * 255).astype(np.uint8))
        dscs.append(compute_segmentation_metrics(pred, gt)['dsc'])
    print(f'SAM ViT-B SNR={snr}: mean DSC = {np.mean(dscs):.4f}')
!zip -r colab_outputs.zip outputs/